<h1>Important</h1>

- The following notebook has only personal learning purposes with no further intention. This was developed using AI tools combined with multiple iterations to refine the code given at first it does generate many errors, from documentation error and more.
- The key intention of this notebook is to show to use a model available from Hugging Face in combination with techniques from the same library to do fine-tuning of the model and show how it works.
- This is not a comercial or industry code to be used, it is just a personal academic learning of how to use different python libraries with ideas of Reinforcement Learning and LLMs.

Key considerations when running this:
- The machine that was used had installed cuda nvidia with 8 GB of capacity, and the idea was to constraint the dataset size and memory usage when training the model.
- Do not load more than 1 model if the CUDA capacity is small because it will lead to potential crushing.
- Try to use cuda and not cpu because is much faster when running.

_____

# Kahneman-Tversky Optimization (KTO)

In [1]:
# Kahneman-Tversky Optimization (KTO) - Standalone Implementation
# Foundation: Nobel Prize-winning Prospect Theory
# Value Function: v(r) = r^α (gains) vs -λ(-r)^β (losses)

import warnings

warnings.filterwarnings("ignore")

import subprocess
import sys
import torch
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
import json
from datetime import datetime

# KTO Configuration
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TEMPERATURE = 0.8
MAX_LENGTH = 512
MAX_NEW_TOKENS = 200
LEARNING_RATE_KTO = 2e-5
NUM_EPOCHS_KTO = 15
BATCH_SIZE_KTO = 2
GRAD_ACCUM_KTO = 8
WARMUP_RATIO = 0.2
LOGGING_STEPS = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

TEST_QUESTIONS = [
    "How do I cook pasta perfectly?",
    "What's the best way to scramble eggs?",
    "How do I make rice that isn't sticky?",
    "What's an easy dinner for beginners?",
    "How do I know when chicken is cooked?",
]


def install_packages():
    packages = [
        "torch>=2.0.0",
        "transformers>=4.36.0",
        "trl>=0.7.4",
        "datasets>=2.14.0",
        "accelerate>=0.21.0",
    ]
    for pkg in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        except:
            pass


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def monitor_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(device) / (1024**3)
        reserved = torch.cuda.memory_reserved(device) / (1024**3)
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        return reserved < 7.5
    return True


def test_model(model, tokenizer, prompt):
    model.eval()
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    )
    if torch.cuda.is_available():
        inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            use_cache=False,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response.strip()


def evaluate_stage(model, tokenizer, stage_name):
    results = {}
    print(f"\n{'='*60}")
    print(f"{stage_name.upper()} MODEL EVALUATION")
    print(f"{'='*60}")

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"\nQ{i}: {question}")
        print("-" * 40)
        response = test_model(model, tokenizer, question)
        results[question] = response
        print(f"Answer: {response}")

    return results


def create_kto_dataset():
    """Create comprehensive KTO dataset with clear preference patterns"""
    examples = [
        # High-quality responses (label=True)
        {
            "prompt": "How do I cook pasta perfectly?",
            "completion": "Perfect pasta requires three key elements: plenty of salted boiling water, proper timing, and pasta water for finishing.\n\nUse a large pot with 4-6 quarts of water per pound of pasta. When it reaches a rolling boil, add 1-2 tablespoons of salt - the water should taste like mild seawater.\n\nAdd pasta and stir immediately to prevent sticking. Cook for 1-2 minutes less than package directions for al dente texture. Taste test frequently in the final minutes.\n\nBefore draining, reserve 1 cup of pasta water. This starchy water helps bind sauces. Never rinse pasta unless making a cold salad.",
            "label": True,
        },
        {
            "prompt": "What's the best way to scramble eggs?",
            "completion": "The secret to creamy scrambled eggs is low heat and constant gentle movement.\n\nWhisk 2-3 eggs with a pinch of salt and 1 tablespoon of butter or cream. Use medium-low heat in a non-stick pan with melted butter.\n\nPour in eggs and let sit for 20-30 seconds. Gently push cooked edges toward center, tilting pan to let raw egg flow underneath. Keep moving gently - never scramble vigorously.\n\nRemove from heat while eggs still look slightly wet. They'll finish cooking from residual heat. Serve immediately on warmed plates.",
            "label": True,
        },
        {
            "prompt": "How do I make rice that isn't sticky?",
            "completion": "Non-sticky rice comes from proper rinsing, correct ratios, and never lifting the lid during cooking.\n\nRinse 1 cup rice in cold water until water runs clear (3-4 rinses). This removes excess starch that causes stickiness.\n\nUse 1 cup rice to 1.5 cups water for white rice. Bring to boil, then immediately reduce to lowest heat. Cover tightly and cook 18 minutes without lifting lid.\n\nAfter cooking, remove from heat but keep covered for 10 more minutes. Then fluff gently with a fork, not a spoon.",
            "label": True,
        },
        # Lower-quality responses (label=False)
        {
            "prompt": "How do I cook pasta perfectly?",
            "completion": "Boil pasta in water until it's soft. Add salt if you want. Drain when done.",
            "label": False,
        },
        {
            "prompt": "What's the best way to scramble eggs?",
            "completion": "Beat eggs and cook them in a pan. Stir them around until they're cooked.",
            "label": False,
        },
        {
            "prompt": "How do I make rice that isn't sticky?",
            "completion": "Use less water and don't overcook it. Rinse the rice first.",
            "label": False,
        },
        # Additional preference pairs for deeper learning
        {
            "prompt": "What's an easy dinner for beginners?",
            "completion": "One-pan chicken and vegetables is perfect for beginners - minimal cleanup and foolproof results.\n\nPreheat oven to 425°F. Season 4 chicken thighs with salt and pepper. Toss halved baby potatoes, chopped carrots, and sliced onion with olive oil, salt, and pepper.\n\nSpread vegetables on large baking sheet, place chicken on top skin-side up. Bake 35-40 minutes until chicken reaches 165°F internal temperature and vegetables are fork-tender.\n\nLet rest 5 minutes before serving. The chicken juices naturally flavor the vegetables.",
            "label": True,
        },
        {
            "prompt": "What's an easy dinner for beginners?",
            "completion": "Cook some chicken and vegetables. Put them in the oven until done.",
            "label": False,
        },
        {
            "prompt": "How do I know when chicken is cooked?",
            "completion": "Food safety is crucial with chicken - use multiple indicators to ensure proper doneness.\n\nUse an instant-read thermometer in the thickest part, avoiding bone. Chicken must reach 165°F internal temperature throughout.\n\nVisually, cooked chicken has no pink color and juices run clear when pierced. The texture should feel firm, not soft or squishy.\n\nWhen in doubt, cook longer. Let chicken rest 5-10 minutes after cooking - internal temperature continues rising.",
            "label": True,
        },
        {
            "prompt": "How do I know when chicken is cooked?",
            "completion": "Cook it until it looks done and isn't pink inside.",
            "label": False,
        },
        {
            "prompt": "How do I make fluffy pancakes?",
            "completion": "Fluffy pancakes require gentle mixing and proper heat control - overmixing is the enemy.\n\nWhisk dry ingredients (2 cups flour, 2 tbsp sugar, 2 tsp baking powder, 1/2 tsp salt) in one bowl. Mix wet ingredients (1 3/4 cups milk, 2 eggs, 1/4 cup melted butter) in another.\n\nPour wet into dry ingredients and stir just until barely combined - lumps are perfectly fine! Overmixing develops gluten, creating tough pancakes.\n\nCook on medium heat, flipping when bubbles form on surface and edges look set.",
            "label": True,
        },
        {
            "prompt": "How do I make fluffy pancakes?",
            "completion": "Mix pancake batter and cook on a pan until both sides are done.",
            "label": False,
        },
        {
            "prompt": "What's the secret to crispy bacon?",
            "completion": "Crispy bacon requires patience and gradual fat rendering - start cold and go slow.\n\nPlace bacon in cold pan without overlapping. Turn heat to medium-low and cook slowly, allowing fat to render gradually over 3-4 minutes before first flip.\n\nThis slow rendering creates crispiness without burning. Total cooking time is 8-12 minutes depending on thickness.\n\nAlternatively, bake at 400°F on rimmed baking sheet for 15-20 minutes - no flipping needed and less splatter.",
            "label": True,
        },
        {
            "prompt": "What's the secret to crispy bacon?",
            "completion": "Cook bacon on high heat and flip it a lot until crispy.",
            "label": False,
        },
    ]
    return examples


def run_kto_training(model, tokenizer):
    print("=" * 80)
    print("KAHNEMAN-TVERSKY OPTIMIZATION (KTO)")
    print("=" * 80)
    print("Theory: v(r) = r^α (gains) vs -λ(-r)^β (losses)")
    print(f"Training: {NUM_EPOCHS_KTO} epochs, LR: {LEARNING_RATE_KTO}")
    print("Foundation: Nobel Prize-winning Prospect Theory for preference learning")

    cleanup_memory()

    kto_data = create_kto_dataset()
    dataset = Dataset.from_list(kto_data)

    print(f"Dataset: {len(kto_data)} preference pairs")
    print(f"Effective batch size: {BATCH_SIZE_KTO * GRAD_ACCUM_KTO}")
    print(f"Loss Aversion: Models learn that losses feel worse than equivalent gains")

    try:
        from trl import KTOTrainer, KTOConfig

        config = KTOConfig(
            output_dir="./temp_kto",
            num_train_epochs=NUM_EPOCHS_KTO,
            per_device_train_batch_size=BATCH_SIZE_KTO,
            gradient_accumulation_steps=GRAD_ACCUM_KTO,
            learning_rate=LEARNING_RATE_KTO,
            max_length=MAX_LENGTH,
            max_prompt_length=MAX_LENGTH // 2,
            logging_steps=LOGGING_STEPS,
            save_strategy="no",
            fp16=False,
            bf16=torch.cuda.is_available(),
            warmup_ratio=WARMUP_RATIO,
            remove_unused_columns=False,
            gradient_checkpointing=True,
            dataloader_drop_last=True,
        )

        trainer = KTOTrainer(
            model=model,
            args=config,
            train_dataset=dataset,
            processing_class=tokenizer,
        )

        print("\nStarting KTO training...")
        print("Mathematical process: Learning human preference psychology patterns")
        trainer.train()
        del trainer

    except Exception as e:
        print(f"KTO training error: {e}")

    cleanup_memory()
    return model, tokenizer


def save_results_json(results, filename):
    os.makedirs("./results", exist_ok=True)
    with open(f"./results/{filename}", "w") as f:
        json.dump(
            {
                "timestamp": datetime.now().isoformat(),
                "model_outputs": results,
                "test_questions": TEST_QUESTIONS,
            },
            f,
            indent=2,
        )


def print_kto_theory():
    """Print comprehensive KTO theoretical foundation"""
    print("\n" + "=" * 80)
    print("KTO THEORETICAL FOUNDATION")
    print("=" * 80)
    print("Mathematical Foundation: Nobel Prize-winning Prospect Theory")
    print()
    print("Value Function:")
    print("  v(r) = r^α           if r ≥ 0  (gains)")
    print("  v(r) = -λ(-r)^β      if r < 0  (losses)")
    print()
    print("Key Psychological Insights:")
    print("  • λ > 1: Loss aversion (losses feel worse than equivalent gains)")
    print("  • α, β < 1: Diminishing sensitivity to magnitude changes")
    print("  • Reference-dependent preferences (not absolute values)")
    print("  • Humans evaluate outcomes relative to expectations")
    print()
    print("Training Process:")
    print("  1. Model learns from positive/negative preference examples")
    print("  2. Incorporates loss aversion into learning dynamics")
    print("  3. Develops reference-dependent quality assessment")
    print("  4. Adapts to human preference psychology patterns")
    print()
    print("Expected Benefits:")
    print("  • Better alignment with human cognitive biases")
    print("  • More intuitive preference learning")
    print("  • Improved understanding of quality gradients")
    print("  • Enhanced response appropriateness")
    print("=" * 80)


def main():
    print("=" * 80)
    print("KAHNEMAN-TVERSKY OPTIMIZATION (KTO) - STANDALONE")
    print("=" * 80)
    print("Behavioral economics approach to preference learning")
    print("Foundation: Nobel Prize-winning Prospect Theory")
    print("Innovation: Loss aversion and reference-dependent preferences")
    print("=" * 80)

    # Print theoretical foundation
    print_kto_theory()

    install_packages()

    print(f"\nInitializing model: {MODEL_NAME}")

    # Load model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )

    # Configure tokenizer
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model.resize_token_embeddings(len(tokenizer))
    monitor_memory()

    # Evaluate base model
    print("\n" + "=" * 50)
    print("EVALUATING BASE MODEL")
    print("=" * 50)
    base_results = evaluate_stage(model, tokenizer, "BASE")
    save_results_json(base_results, "kto_base_results.json")

    # Run KTO training
    print("\n" + "=" * 50)
    print("STARTING KTO TRAINING")
    print("=" * 50)
    model, tokenizer = run_kto_training(model, tokenizer)

    # Evaluate trained model
    print("\n" + "=" * 50)
    print("EVALUATING TRAINED MODEL")
    print("=" * 50)
    trained_results = evaluate_stage(model, tokenizer, "KTO")
    save_results_json(trained_results, "kto_trained_results.json")

    # Save final model
    print(f"\nSaving KTO-trained model...")
    os.makedirs("./models/kto_standalone", exist_ok=True)
    model.save_pretrained("./models/kto_standalone")
    tokenizer.save_pretrained("./models/kto_standalone")

    # Compare results
    print(f"\n{'='*80}")
    print("COMPARATIVE ANALYSIS: Base vs KTO")
    print(f"{'='*80}")
    print("Focus: How behavioral economics improves preference learning")
    print("Key aspects: Loss aversion, reference-dependent quality assessment")
    print(f"{'='*80}")

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"\n[QUESTION {i}]: {question}")
        print("=" * 60)
        print(f"\n[BASE MODEL (No preference learning)]:")
        print(f"{base_results[question]}")
        print(f"\n[KTO MODEL (Behavioral economics preferences)]:")
        print(f"{trained_results[question]}")
        print("=" * 60)

    # Analysis summary
    print(f"\n{'='*80}")
    print("KTO TRAINING ANALYSIS")
    print(f"{'='*80}")
    print("Theoretical Achievements:")
    print("  • Applied Kahneman-Tversky Prospect Theory to LLM training")
    print("  • Incorporated loss aversion into preference optimization")
    print("  • Learned reference-dependent quality assessment")
    print("  • Developed human-like cognitive bias patterns")
    print()
    print("Expected Improvements:")
    print("  • Better alignment with human preference psychology")
    print("  • More intuitive quality gradients in responses")
    print("  • Enhanced understanding of relative vs absolute quality")
    print("  • Improved sensitivity to user satisfaction patterns")
    print()
    print("Mathematical Process:")
    print("  • Value function models human perception of gains/losses")
    print("  • Loss aversion parameter λ > 1 amplifies negative feedback")
    print("  • Diminishing sensitivity captures human decision patterns")
    print("  • Reference-dependent evaluation mirrors cognitive biases")
    print(f"{'='*80}")

    print(f"\n{'='*80}")
    print("KTO TRAINING COMPLETED")
    print(f"{'='*80}")
    print("Results saved to:")
    print("  • ./models/kto_standalone/ - Trained model")
    print("  • ./results/kto_base_results.json - Base evaluation")
    print("  • ./results/kto_trained_results.json - Trained evaluation")
    print()
    print("Key Innovation:")
    print("  KTO applies Nobel Prize-winning behavioral economics theory")
    print("  to create more human-aligned AI preference learning")
    print(f"{'='*80}")

In [2]:
# Execute main
if __name__ == "__main__":
    main()

KAHNEMAN-TVERSKY OPTIMIZATION (KTO) - STANDALONE
Behavioral economics approach to preference learning
Foundation: Nobel Prize-winning Prospect Theory
Innovation: Loss aversion and reference-dependent preferences

KTO THEORETICAL FOUNDATION
Mathematical Foundation: Nobel Prize-winning Prospect Theory

Value Function:
  v(r) = r^α           if r ≥ 0  (gains)
  v(r) = -λ(-r)^β      if r < 0  (losses)

Key Psychological Insights:
  • λ > 1: Loss aversion (losses feel worse than equivalent gains)
  • α, β < 1: Diminishing sensitivity to magnitude changes
  • Reference-dependent preferences (not absolute values)
  • Humans evaluate outcomes relative to expectations

Training Process:
  1. Model learns from positive/negative preference examples
  2. Incorporates loss aversion into learning dynamics
  3. Develops reference-dependent quality assessment
  4. Adapts to human preference psychology patterns

Expected Benefits:
  • Better alignment with human cognitive biases
  • More intuitive pref

`torch_dtype` is deprecated! Use `dtype` instead!


GPU Memory: 0.92GB allocated, 1.19GB reserved

EVALUATING BASE MODEL

BASE MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: Cooking pasta to perfection can be achieved by following a few simple steps. Here's how:

1. Choose the right type of pasta: There are many types of pasta, each with its own unique texture and flavor. For example, spaghetti is very thin and has a smooth texture, while fettuccine has a more substantial texture.

2. Rinse your pasta: Before cooking, rinse your pasta under cold water to remove any excess moisture. This will help prevent sticking and make the pasta easier to cook.

3. Add enough water: The amount of water you use will depend on the size of your pasta. A general rule of thumb is to add about 4 cups (960 ml) of boiling water for every pound (500 g) of pasta. However, you may need to adjust this depending on the thickness of your pasta.

4. Cook the pasta: Bring a large pot of salted water to a boil. A

Extracting prompt from train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting KTO training...
Mathematical process: Learning human preference psychology patterns


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
5,0.320300
10,0.033900
15,0.038500



EVALUATING TRAINED MODEL

KTO MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: Perfect pasta requires three key elements: plenty of salted boiling water, proper timing, and pasta water for finishing.

Use a large pot with 4-6 quarts of water per pound of pasta. When it reaches a rolling boil, add 1-2 tablespoons of salt - the water should taste like mild seawater.

Add pasta and stir immediately to prevent sticking. Cook for 1-2 minutes less than package directions for al dente texture. Taste test frequently in the final minutes.

Before draining, reserve 1 cup of pasta water. This starchy water helps bind sauces. Never rinse pasta unless making a cold salad.

Q2: What's the best way to scramble eggs?
----------------------------------------
Answer: The secret to creamy scrambled eggs is low heat and constant gentle movement.

Whisk 2-3 eggs with a pinch of salt and 1 tablespoon of butter or cream. Use medium-low heat in a non-stick